In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [4]:
np.random.seed(42)
n = 1000

In [5]:

sqft = np.random.randint(800, 5000, n)
bedrooms = np.random.randint(1, 6, n)

In [6]:

noise = np.random.normal(0, 20000, n)
price = 50 * sqft + 30000 * bedrooms + 80000 + noise
price = price.astype(int)

In [7]:

addresses = [f"{i} Main St, Portland, OR {97000 + (i % 100):05d}" for i in range(n)]

In [8]:
df = pd.DataFrame({
    "price": price,
    "sqft": sqft,
    "bedrooms": bedrooms,
    "abbreviatedAddress": addresses
})

print("Shape:", df.shape)

Shape: (1000, 4)


In [9]:

df.to_csv("/tmp/portland_housing.csv", index=False)
df = pd.read_csv("/tmp/portland_housing.csv", low_memory=False)
print("Dataset loaded successfully")

Dataset loaded successfully


In [10]:
print(df.head())

    price  sqft  bedrooms             abbreviatedAddress
0  237646  1660         2  0 Main St, Portland, OR 97000
1  433834  4572         5  1 Main St, Portland, OR 97001
2  313456  3892         1  2 Main St, Portland, OR 97002
3  197418  1266         1  3 Main St, Portland, OR 97003
4  316058  4244         1  4 Main St, Portland, OR 97004


In [11]:
print(df.dtypes)

price                  int64
sqft                   int64
bedrooms               int64
abbreviatedAddress    object
dtype: object


In [12]:
print(df.columns.tolist())

['price', 'sqft', 'bedrooms', 'abbreviatedAddress']


In [13]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   price               1000 non-null   int64 
 1   sqft                1000 non-null   int64 
 2   bedrooms            1000 non-null   int64 
 3   abbreviatedAddress  1000 non-null   object
dtypes: int64(3), object(1)
memory usage: 31.4+ KB
None


In [14]:
print(df.describe())

               price         sqft     bedrooms
count    1000.000000  1000.000000  1000.000000
mean   321073.286000  2984.633000     3.032000
std     73611.743005  1185.574005     1.430742
min    137082.000000   803.000000     1.000000
25%    267356.750000  1966.750000     2.000000
50%    323313.500000  2989.000000     3.000000
75%    373613.500000  3983.750000     4.000000
max    514508.000000  4999.000000     5.000000


In [15]:

df['zip_extracted'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: re.findall(r'\d{5}', x)[0] if re.findall(r'\d{5}', x) else None
)
print("Columns:", df.columns.tolist())
print(df['zip_extracted'].head(10))

Columns: ['price', 'sqft', 'bedrooms', 'abbreviatedAddress', 'zip_extracted']
0    97000
1    97001
2    97002
3    97003
4    97004
5    97005
6    97006
7    97007
8    97008
9    97009
Name: zip_extracted, dtype: object


In [16]:

df['clean_address'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: re.sub(r'[^a-zA-Z0-9 ]', '', x)
)
print(df['clean_address'].head())

0    0 Main St Portland OR 97000
1    1 Main St Portland OR 97001
2    2 Main St Portland OR 97002
3    3 Main St Portland OR 97003
4    4 Main St Portland OR 97004
Name: clean_address, dtype: object


In [17]:

df['addr_number'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: re.findall(r'^\d+', x)
)
df['addr_number'] = df['addr_number'].apply(lambda x: x[0] if x else None)
print(df['addr_number'].head())

0    0
1    1
2    2
3    3
4    4
Name: addr_number, dtype: object


In [18]:

df['has_direction'] = df['abbreviatedAddress'].astype(str).apply(
    lambda x: bool(re.search(r'\b(NE|NW|SE|SW)\b', x))
)
print(df['has_direction'].value_counts())

has_direction
False    1000
Name: count, dtype: int64


In [19]:
print('Null counts:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

Null counts:
price                 0
sqft                  0
bedrooms              0
abbreviatedAddress    0
zip_extracted         0
clean_address         0
addr_number           0
has_direction         0
dtype: int64


In [20]:

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
print('Nulls after filling:', df.isnull().sum().sum())

Nulls after filling: 0


In [21]:
print(df.isnull().any())

price                 False
sqft                  False
bedrooms              False
abbreviatedAddress    False
zip_extracted         False
clean_address         False
addr_number           False
has_direction         False
dtype: bool


In [22]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('After removing duplicates:', df.shape)

Duplicate rows: 0
After removing duplicates: (1000, 8)


In [23]:

df.columns = df.columns.str.strip().str.lower()
print(df.columns.tolist())

['price', 'sqft', 'bedrooms', 'abbreviatedaddress', 'zip_extracted', 'clean_address', 'addr_number', 'has_direction']


In [24]:

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df[['price', 'sqft', 'bedrooms']].fillna(
    df[['price', 'sqft', 'bedrooms']].median(), inplace=True
)
print('Shape after formatting:', df.shape)

Shape after formatting: (1000, 8)


In [25]:
bins = [0, 1000, 2000, 3000, 6000]
labels = ['Small', 'Medium', 'Large', 'Very Large']
df['area_bin'] = pd.cut(df['sqft'], bins=bins, labels=labels)
print(df['area_bin'].value_counts())

area_bin
Very Large    495
Large         250
Medium        215
Small          40
Name: count, dtype: int64


In [26]:
price_bins = [0, 150000, 300000, 500000, 1000000]
price_labels = ['Budget', 'Mid', 'Upper-Mid', 'Luxury']
df['price_bin'] = pd.cut(df['price'], bins=price_bins, labels=price_labels)
print(df['price_bin'].value_counts())

price_bin
Upper-Mid    614
Mid          381
Budget         4
Luxury         1
Name: count, dtype: int64


In [27]:
print(df.describe())

               price         sqft     bedrooms
count    1000.000000  1000.000000  1000.000000
mean   321073.286000  2984.633000     3.032000
std     73611.743005  1185.574005     1.430742
min    137082.000000   803.000000     1.000000
25%    267356.750000  1966.750000     2.000000
50%    323313.500000  2989.000000     3.000000
75%    373613.500000  3983.750000     4.000000
max    514508.000000  4999.000000     5.000000


In [28]:
print('Mean price:', df['price'].mean())
print('Median price:', df['price'].median())
print('Std price:', df['price'].std())

Mean price: 321073.286
Median price: 323313.5
Std price: 73611.74300547775


In [29]:
print('Skewness:\n', df.skew(numeric_only=True))

Skewness:
 price           -0.027564
sqft            -0.093820
bedrooms        -0.019298
has_direction    0.000000
dtype: float64


In [30]:
avg_price_bed = df.groupby('bedrooms')['price'].mean()
print(avg_price_bed)

bedrooms
1    264898.569231
2    291157.191388
3    319285.994382
4    349869.746341
5    375633.835681
Name: price, dtype: float64


In [31]:
avg_area = df.groupby('bedrooms')['sqft'].mean()
print(avg_area)

bedrooms
1    3040.497436
2    3014.708134
3    2956.820225
4    2996.756098
5    2915.553991
Name: sqft, dtype: float64


In [32]:
grouped = df.groupby('bedrooms')['price'].agg(['mean', 'min', 'max'])
print(grouped)

                   mean     min     max
bedrooms                               
1         264898.569231  137082  397063
2         291157.191388  158209  427307
3         319285.994382  188243  455166
4         349869.746341  218782  478161
5         375633.835681  258315  514508


In [33]:
print("Final dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Final dataset shape: (1000, 10)
Columns: ['price', 'sqft', 'bedrooms', 'abbreviatedaddress', 'zip_extracted', 'clean_address', 'addr_number', 'has_direction', 'area_bin', 'price_bin']
